# HearAssist — Fine-tuning de Whisper medium sur la darija, avec augmentation

Objectif : faire mieux que `whisper-small` fine-tuné (WER 39,2 %, qui surapprenait) et que `ychafiqui` (WER 33,1 %),
grâce au bruit de fond (ESC-50), à SpecAugment et au dropout.

**Réglages Kaggle (panneau de droite) :** Accelerator = **GPU T4 x2** · Internet = **On** ·
Input = la **dernière version** du dataset contenant `darija_asr`.

**Comment l'utiliser :**
1. Exécute les cellules des parties 1 à 4 une par une. Le test rapide doit afficher `Modèle final enregistré`.
2. Lance ensuite **Save Version → Save & Run All (Commit)**. Tu peux fermer ton navigateur.
3. Récupère le modèle et les scores dans l'onglet **Output** de la version terminée.

## 1. Réglages
Réglé sur `medium`. Les poids, les gradients et l'optimiseur occupent déjà environ 11,5 Go du T4 : `BATCH, ACCUM = 8, 2` et `4, 4` saturent sa mémoire, on utilise donc `2, 8`.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # limite la mémoire GPU perdue
BASE = "openai/whisper-medium"
BATCH, ACCUM = 2, 8
OUT = "/kaggle/working/whisper-medium-darija-aug"
MAX_STEPS = 2000

## 2. Préparation

In [ ]:
import os, glob, json, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # un seul GPU : plus simple et plus stable
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers>=5.13" "datasets>=3.0" "peft>=0.13" accelerate jiwer soundfile scipy ctranslate2 faster-whisper tiktoken sentencepiece
# torchao préinstallé sur Kaggle (trop ancien) fait planter peft ; on n'en a pas besoin
!pip uninstall -y -q torchao

In [ ]:
# Copie du code (dataset Kaggle contenant darija_asr/) vers le dossier de travail
src = glob.glob("/kaggle/input/**/finetune_whisper.py", recursive=True)
assert src, "Ajoute le dataset contenant darija_asr (Add Input)"
CODE = "/kaggle/working/darija_asr"
shutil.copytree(os.path.dirname(src[0]), CODE, dirs_exist_ok=True)
os.chdir(CODE)
assert "save-only-model" in open("finetune_whisper.py", encoding="utf-8").read(), "Ancienne version du code : mets à jour le dataset"
print("Code prêt dans", CODE)

## 3. Bruits de fond (ESC-50)
2 000 sons d'environnement, environ 600 Mo. Licence CC BY-NC : à citer dans le rapport.

In [ ]:
!mkdir -p /kaggle/temp
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /kaggle/temp/esc50.zip
!unzip -q -o /kaggle/temp/esc50.zip -d /kaggle/temp/
NOISE_DIR = "/kaggle/temp/ESC-50-master/audio"
n_noise = len(glob.glob(f"{NOISE_DIR}/*.wav"))
assert n_noise, "ESC-50 non trouvé : vérifie qu'Internet est activé"
print(n_noise, "fichiers de bruit")

## 4. Test rapide (environ 2 minutes)
Vérifie que le bruit, SpecAugment et le dropout fonctionnent. Le score n'a aucune valeur ici.
En cas d'erreur, ne lance pas la suite.

In [ ]:
!python finetune_whisper.py --base-model openai/whisper-tiny --max-train-samples 64 --eval-samples 16 \
    --max-steps 10 --eval-steps 10 --warmup-steps 0 --noise-dir {NOISE_DIR} --spec-augment --dropout 0.1 --save-only-model \
    --output-dir /kaggle/working/smoke_aug
assert os.path.exists("/kaggle/working/smoke_aug/final"), "Le test rapide a échoué : regarde l'erreur ci-dessus"
shutil.rmtree("/kaggle/working/smoke_aug")
print("Test rapide réussi")

## 5. Fine-tuning réel
Durée mesurée : environ 13 s par étape, évaluations comprises, soit environ 7 h 10 pour 2 000 étapes (plus environ 1 h 30 d'évaluation finale).
Le meilleur checkpoint (WER le plus bas) est conservé automatiquement.

`/kaggle/working` est limité à environ 20 Go : la cellule suivante supprime les restes d'une tentative précédente, et `--save-only-model` réduit chaque checkpoint d'environ 9 Go à 3 Go.

In [ ]:
# Supprime les restes des tentatives précédentes, puis affiche l'espace disque (il faut au moins 15 Go libres)
shutil.rmtree(OUT, ignore_errors=True)
shutil.rmtree("/kaggle/working/smoke_aug", ignore_errors=True)
!du -sh /kaggle/working/* 2>/dev/null
!df -h /kaggle/working

In [ ]:
!python finetune_whisper.py --base-model {BASE} --output-dir {OUT} \
    --batch-size {BATCH} --grad-accum {ACCUM} --max-steps {MAX_STEPS} --eval-steps 250 --eval-samples 200 \
    --noise-dir {NOISE_DIR} --noise-prob 0.3 --spec-augment --dropout 0.1 --save-only-model
assert os.path.exists(f"{OUT}/final"), "L'entraînement a échoué : regarde l'erreur ci-dessus"

## 6. Conversion et évaluation finale
- **200 clips** : à comparer aux modèles de référence (`ychafiqui` : WER 33,1 %).
- **Toute la validation** (1 065 clips) : le chiffre à mettre dans le rapport.

In [ ]:
!python convert_ct2.py --model {OUT}/final --out {OUT}/ct2 --quantization float16
!python eval_asr.py --backend faster-whisper --model {OUT}/ct2 --max-samples 200 --out-dir {OUT}/eval
!python eval_asr.py --backend faster-whisper --model {OUT}/ct2 --max-samples 0 --out-dir {OUT}/eval

In [ ]:
for f in sorted(glob.glob(f"{OUT}/eval/*_metrics.json")):
    m = json.load(open(f))
    print(f'{m["n_samples"]} clips | WER {m["wer"]:.1%} | CER {m["cer"]:.1%} | {m["seconds_per_clip"]} s/clip')

## 7. Nettoyage
Garde seulement `final/`, `ct2/` et `eval/` pour alléger l'onglet Output.

In [ ]:
for d in glob.glob(f"{OUT}/checkpoint-*"):
    shutil.rmtree(d)
!du -sh {OUT}/*